# Data Preprocessing
**Mục tiêu**: Biến đổi dữ liệu thô ban đầu thành dữ liệu có định dạng chuẩn, sạch sẽ, có ý nghĩa và đạt chất lượng tốt nhất trước khi đưa vào học máy

In [1]:
# Import thư viện
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


### Bước 1: Đọc và làm sạch dữ liệu
- **Mục tiêu:** Tải dữ liệu từ tệp `car_purchasing.csv` lên bộ nhớ để có thể xử lý bằng thư viện Pandas.


In [2]:

df = pd.read_csv('../data/car_purchasing.csv')

print("First 5 rows of initial data:")
print(df.head(), "\n")


First 5 rows of initial data:
   gender        age  annual Salary  credit card debt    net worth  \
0       0  41.851720    62812.09301      11609.380910  238961.2505   
1       0  40.870623    66646.89292       9572.957136  530973.9078   
2       1  43.152897    53798.55112      11160.355060  638467.1773   
3       1  58.271369    79370.03798      14426.164850  548599.0524   
4       1  57.313749    59729.15130       5358.712177  560304.0671   

   car purchase amount  
0          35321.45877  
1          45115.52566  
2          42925.70921  
3          67422.36313  
4          55915.46248   



### Bước 2: Tách features (X) và target (y)
- **Mục tiêu:** Chia dữ liệu thành phần "Câu hỏi" (X) và "Đáp án" (y).
- **Giải thích:** - Máy học theo cơ chế có giám sát (Supervised Learning) cần có đầu vào và đáp án mục tiêu. 
  - Biến `X`: Tuổi, Mức lương, Nợ thẻ tín dụng, Tài sản.
  - Biến `y`: `car purchase amount` (Số tiền mua xe) để làm mục tiêu dự đoán.

In [3]:
X = df.drop('car purchase amount', axis=1)
y = df['car purchase amount']

### Bước 3: Chia dữ liệu
- **Mục tiêu:** Chia toàn bộ tập dữ liệu gốc thành 3 tập riêng biệt với tỷ lệ 80% cho Train - 10% cho Validation - 10% cho Test.
- **Ý nghĩa các tập:** 
  - **Train (80%):** Dành cho mô hình "học".
  - **Validation (10%):** Giúp đánh giá nhanh các mô hình, giúp tinh chỉnh tham số và tránh học vẹt (Overfitting)
  - **Test (10%)**: Để đánh giá tính hiệu quả của mô hình một cách khách quan nhất


In [4]:
# Lần cắt 1: Lấy 80% cho Train, 20% còn lại gộp chung vào tập Temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Lần cắt 2: Cắt đôi tập Temp (20%) thành 10% Validation và 10% Test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

### Bước 4: Chuẩn hóa dữ liệu
- **Mục tiêu:** Đưa tất cả các cột dữ liệu có thang đo chênh lệch lớn về cùng một hệ quy chiếu là khoảng [0, 1].
- **Giải thích:** Nếu không chuẩn hóa, các mô hình học máy (đặc biệt là mạng Noron) sẽ bị thiên lệch và xử lý sai lệch do các con số quá lơn


In [5]:
scaler_X = MinMaxScaler()

# 1. Chỉ Dùng fit_transform cho tập Train
X_train_scaled = scaler_X.fit_transform(X_train)

# 2. Dùng transform cho cả tập Validation và tập Test
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

# Chuẩn hóa cột đáp án (y) tương tự
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

### In kết quả kiểm tra sau khi chuẩn hóa

In [6]:
print("--- PREPROCESSING COMPLETED ---")
print(f"Number of customers for Train (80%): {X_train_scaled.shape[0]}")
print(f"Number of customers for Validation (10%): {X_val_scaled.shape[0]}")
print(f"Number of customers for Test (10%): {X_test_scaled.shape[0]}")
print("\nData of the first customer in the Train set after normalization:")
print(X_train_scaled[0])

--- PREPROCESSING COMPLETED ---
Number of customers for Train (80%): 400
Number of customers for Validation (10%): 50
Number of customers for Test (10%): 50

Data of the first customer in the Train set after normalization:
[0.         0.3251457  0.62918035 0.48860397 0.62206021]


### Bước 5: Xuất dữ liệu đã được chuẩn hóa thành file CSV
- **Mục tiêu:** Lưu kết quả sau khi xử lý thành các file độc lập để phục vụ cho việc huấn luyện học máy tiếp theo

In [ ]:
# 1. Chuyển đổi các mảng (numpy array) từ kết quả của MinMaxScaler về lại DataFrame
# X.columns giúp giữ lại tên của các cột đặc trưng (features) ban đầu
df_train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
df_val_final = pd.DataFrame(X_val_scaled, columns=X.columns)
df_test_final = pd.DataFrame(X_test_scaled, columns=X.columns)

# 2. Gắn thêm cột mục tiêu (y) đã chuẩn hóa vào các DataFrame tương ứng
df_train_final['car purchase amount'] = y_train_scaled
df_val_final['car purchase amount'] = y_val_scaled
df_test_final['car purchase amount'] = y_test_scaled

# 3. Lưu thành 3 file CSV riêng biệt
# Tham số index=False giúp bỏ đi cột số thứ tự mặc định của Pandas khi lưu file
df_train_final.to_csv('../data/preprocessed/train.csv', index=False)
df_val_final.to_csv('../data/preprocessed/val.csv', index=False)
df_test_final.to_csv('../data/preprocessed/test.csv', index=False)

print("\n--- FILE EXPORTED SUCCESSFULLY ---")
print("Saved 3 files: train.csv, val.csv và test.csv.")


--- XUẤT FILE THÀNH CÔNG ---
Đã lưu 3 file: train.csv, val.csv và test.csv.
